# 🧪 Phase 4: Advanced Deep Learning Models (BERT & RoBERTa)

> **⚠️ COMPUTATION WARNING:**
> Fine-tuning Transformer models like BERT and RoBERTa requires immense computational power. If you are running this on a standard laptop without a dedicated NVIDIA GPU, this notebook will take hours or days to complete.
> 
> **Recommendation:** Upload this notebook and your `data/processed/` folder to **Google Colab** and enable the T4 GPU runtime (`Runtime > Change runtime type > T4 GPU`) to train these models in minutes.

---

## 1. Environment Setup

In [ ]:
# Run this cell if you are in Google Colab to install dependencies
!pip install -q transformers torch datasets scikit-learn pandas

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# Ensure GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Dataset Preparation
We will load our cleaned datasets. For demonstration purposes and fast iteration, we will sample the dataset. You can increase the sample size for final portfolio results.

In [ ]:
# Load data (Update path if running on Google Colab Drive)
try:
    df_news = pd.read_csv('../data/processed/isot_cleaned.csv')
    df_tweets = pd.read_csv('../data/processed/covid_tweets_cleaned.csv')
except FileNotFoundError:
    # Fallback for Colab if uploaded directly
    print("Please ensure 'isot_cleaned.csv' and 'covid_tweets_cleaned.csv' are uploaded.")
    df_news = pd.read_csv('isot_cleaned.csv')
    df_tweets = pd.read_csv('covid_tweets_cleaned.csv')

# --- SAMPLE SIZES ---
# Change these to len(df) to train on everything
NEWS_SAMPLE = 2000 
TWEET_SAMPLE = 2000

df_news = df_news.sample(n=NEWS_SAMPLE, random_state=42).reset_index(drop=True)
df_tweets = df_tweets.sample(n=TWEET_SAMPLE, random_state=42).reset_index(drop=True)

print(f"News dataset: {len(df_news)} samples")
print(f"Tweet dataset: {len(df_tweets)} samples")

---
## 3. Fine-Tuning BERT for News Articles
We use `bert-base-uncased` for the long-form news articles.

In [ ]:
news_model_name = "bert-base-uncased"
news_tokenizer = AutoTokenizer.from_pretrained(news_model_name)

def tokenize_news(batch):
    return news_tokenizer(batch["text_clean"], padding="max_length", truncation=True, max_length=256)

# Prepare Dataset
news_ds = Dataset.from_pandas(df_news[['text_clean', 'label_binary']])
news_ds = news_ds.rename_column("label_binary", "labels")
news_ds = news_ds.map(tokenize_news, batched=True)

# Train/Test Split
news_split = news_ds.train_test_split(test_size=0.2, seed=42)
train_news_ds = news_split['train']
test_news_ds = news_split['test']

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Load Model
news_model = AutoModelForSequenceClassification.from_pretrained(news_model_name, num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir="./results_news",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
)

news_trainer = Trainer(
    model=news_model,
    args=training_args,
    train_dataset=train_news_ds,
    eval_dataset=test_news_ds,
    compute_metrics=compute_metrics,
)

In [ ]:
news_trainer.train() # Uncomment this line to execute training!
print(news_trainer.evaluate())

---
## 4. Fine-Tuning RoBERTa for Tweets
We use `cardiffnlp/twitter-roberta-base`, a model pre-trained on 58M tweets. It natively understands hashtags, handles '@' mentions, and parses Twitter-specific semantics much better than standard BERT.

In [ ]:
tweet_model_name = "cardiffnlp/twitter-roberta-base"
tweet_tokenizer = AutoTokenizer.from_pretrained(tweet_model_name)

def tokenize_tweets(batch):
    # Tweets are short, so max_length 128 is plenty
    return tweet_tokenizer(batch["text_clean"], padding="max_length", truncation=True, max_length=128)

# Prepare Dataset
tweet_ds = Dataset.from_pandas(df_tweets[['text_clean', 'label_binary']])
tweet_ds = tweet_ds.rename_column("label_binary", "labels")
tweet_ds = tweet_ds.map(tokenize_tweets, batched=True)

# Train/Test Split
tweet_split = tweet_ds.train_test_split(test_size=0.2, seed=42)
train_tweet_ds = tweet_split['train']
test_tweet_ds = tweet_split['test']

In [ ]:
# Load Model
tweet_model = AutoModelForSequenceClassification.from_pretrained(tweet_model_name, num_labels=2).to(device)

training_args_tweet = TrainingArguments(
    output_dir="./results_tweets",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

tweet_trainer = Trainer(
    model=tweet_model,
    args=training_args_tweet,
    train_dataset=train_tweet_ds,
    eval_dataset=test_tweet_ds,
    compute_metrics=compute_metrics,
)

In [ ]:
tweet_trainer.train() # Uncomment this line to execute training!
print(tweet_trainer.evaluate())